In [ ]:
%%bash
set -Eeuo pipefail

export DEBIAN_FRONTEND=noninteractive

# ---- Configuration ----
NDK_VERSION="r27c"
ANDROID_API="android-28"
ANDROID_ABI="arm64-v8a"
LLAMA_REF="master"

WORK="/content/llama-android-build"
SRC="${WORK}/llama.cpp"
BUILD="${WORK}/build"
OUTPUT="/content/libllama_server.so"

NDK="/content/android-ndk-${NDK_VERSION}"
TOOLBIN="${NDK}/toolchains/llvm/prebuilt/linux-x86_64/bin"

# ---- Install tools ----
apt-get update
apt-get install -y git cmake ninja-build unzip

# ---- Download NDK if needed ----
if [ ! -f "${NDK}/build/cmake/android.toolchain.cmake" ]; then
    wget -q --show-progress \
      "https://dl.google.com/android/repository/android-ndk-${NDK_VERSION}-linux.zip" \
      -O "/content/android-ndk.zip"
    unzip -q /content/android-ndk.zip -d /content
    rm -f /content/android-ndk.zip
fi

export ANDROID_NDK="${NDK}"

# ---- Clone llama.cpp ----
mkdir -p "${WORK}"
if [ ! -d "${SRC}/.git" ]; then
    git clone --depth 1 --branch "${LLAMA_REF}" \
      https://github.com/ggml-org/llama.cpp "${SRC}"
fi

# ---- Clean old configuration ----
rm -rf "${BUILD}"
mkdir -p "$(dirname "${OUTPUT}")"

# ---- Configure size-optimized build ----
cmake -S "${SRC}" -B "${BUILD}" -G Ninja \
    -DCMAKE_TOOLCHAIN_FILE="${NDK}/build/cmake/android.toolchain.cmake" \
    -DANDROID_ABI="${ANDROID_ABI}" \
    -DANDROID_PLATFORM="${ANDROID_API}" \
    -DANDROID_STL=c++_static \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_C_FLAGS_RELEASE="-Oz -DNDEBUG -ffunction-sections -fdata-sections" \
    -DCMAKE_CXX_FLAGS_RELEASE="-Oz -DNDEBUG -ffunction-sections -fdata-sections" \
    -DCMAKE_EXE_LINKER_FLAGS="-Wl,--gc-sections" \
    -DCMAKE_INTERPROCEDURAL_OPTIMIZATION=ON \
    -DBUILD_SHARED_LIBS=OFF \
    -DGGML_NATIVE=OFF \
    -DGGML_OPENMP=OFF \
    -DGGML_LLAMAFILE=OFF \
    -DGGML_CUDA=OFF \
    -DGGML_VULKAN=OFF \
    -DGGML_SYCL=OFF \
    -DGGML_METAL=OFF \
    -DLLAMA_CURL=OFF \
    -DLLAMA_OPENSSL=OFF \
    -DLLAMA_BUILD_TESTS=OFF \
    -DLLAMA_BUILD_EXAMPLES=OFF \
    -DLLAMA_BUILD_SERVER=ON

# ---- Build executable ----
cmake --build "${BUILD}" \
    --target llama-server --parallel 2

BIN="${BUILD}/bin/llama-server"

if [ ! -f "${BIN}" ]; then
    echo "ERROR: llama-server executable not found"
    find "${BUILD}" -type f -name 'llama-server' -print
    exit 1
fi

# ---- Strip symbols ----
"${TOOLBIN}/llvm-strip" --strip-unneeded "${BIN}"

# ---- Export under your app's expected name ----
cp "${BIN}" "${OUTPUT}"
chmod 755 "${OUTPUT}"

echo
echo "===== BUILD COMPLETE ====="
file "${OUTPUT}"
ls -lh "${OUTPUT}"
du -h "${OUTPUT}"
echo
echo "Android ABI: ${ANDROID_ABI}"
echo "Android API: ${ANDROID_API}"